In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


2025-12-12 09:26:09.187904: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-12 09:26:09.365931: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-12 09:26:20.254980: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
import json
import torch
from torch.utils.data import Dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

# 1) Charger les données JSON
with open("/mnt/d/Etude/Master cours/IA Generative/environnement/projet-aisca/test-steven/data.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

texts = [item["text"] for item in raw_data]
labels = [item["label"] for item in raw_data]

# 2) Préparer le tokenizer
model_name = "bert-base-multilingual-cased"  # mieux pour le français
tokenizer = BertTokenizerFast.from_pretrained(model_name)

encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128
)

# 3) Dataset PyTorch
class JsonBusinessDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = JsonBusinessDataset(encodings, labels)

# 4) Modèle BERT pour classification
num_labels = len(set(labels))
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# 5) Paramètres d'entraînement
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10
)

# 6) Trainer Hugging Face
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

# 7) Sauvegarde du modèle et du tokenizer
model.save_pretrained("./bert-metier")
tokenizer.save_pretrained("./bert-metier")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss


('./bert-metier/tokenizer_config.json',
 './bert-metier/special_tokens_map.json',
 './bert-metier/vocab.txt',
 './bert-metier/added_tokens.json',
 './bert-metier/tokenizer.json')

In [4]:
from transformers.pipelines import pipeline


def utiliser_bert():
    # 1. Chargement du pipeline
    # On demande la tâche "fill-mask" (remplir le masque)
    # On utilise 'bert-base-multilingual-cased' qui comprend le français
    print("Chargement du modèle BERT en cours...")
    unmasker = pipeline('fill-mask', model='bert-base-multilingual-cased')

    # 2. La phrase à tester
    # Note le token spécial [MASK] que BERT doit deviner.
    phrase = "Le médecin a prescrit un [MASK] au patient."

    print(f"\nPhrase d'entrée : {phrase}")
    print("-" * 30)

    # 3. Prédiction
    resultats = unmasker(phrase)

    # 4. Affichage des résultats
    # BERT nous donne plusieurs propositions avec un score de confiance
    for i, res in enumerate(resultats):
        mot_predit = res['token_str']
        score = res['score'] * 100 # En pourcentage
        phrase_complete = res['sequence']

        print(f"Option {i+1}: {mot_predit} ({score:.2f}%)")
        print(f"   -> {phrase_complete}")

if __name__ == "__main__":
    utiliser_bert()

Chargement du modèle BERT en cours...


Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0



Phrase d'entrée : Le médecin a prescrit un [MASK] au patient.
------------------------------
Option 1: médecin (11.05%)
   -> Le médecin a prescrit un médecin au patient.
Option 2: diagnostic (6.82%)
   -> Le médecin a prescrit un diagnostic au patient.
Option 3: traitement (6.12%)
   -> Le médecin a prescrit un traitement au patient.
Option 4: rapport (3.09%)
   -> Le médecin a prescrit un rapport au patient.
Option 5: secours (2.39%)
   -> Le médecin a prescrit un secours au patient.
